# 10 · Testing with pytest

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 2 | about 60 minutes live | [09 · Exceptions and logging](09_exceptions_and_logging.ipynb) | [11 · From notebook to package](11_from_notebook_to_package.ipynb) |

The warm-up notebook had a bug that never crashed: next month's customers were encoded differently from the
training data, and the predictions were silently wrong. You only find bugs like that if something checks the
behaviour on purpose, every time the code changes. That something is an **automated test**.

A test is a small function that runs a piece of your code on known input and checks the result. A test suite
is a collection of them that you run with one command before every commit. When a test that used to pass fails,
you know immediately which change broke what.

> **Learning goals.** By the end you can:
>
> - write and run tests with pytest from the terminal, and read its output
> - use `pytest.approx`, `pytest.raises`, fixtures and `parametrize` to keep tests short
> - decide what is worth testing in data science code, and what is not
> - turn a bug into a regression test before fixing it

**Before you start:** notebooks 04 (transformers) and 09 (exceptions). You should have run `uv sync`, so the
reference package `churn` (in `src/churn`) can be imported.

**In this notebook**

- Setup
- Theory 1 · Tests live in files, pytest runs them
- Theory 2 · A first test: Arrange, Act, Assert
- Theory 3 · Comparing floats and expecting errors
- Hands-on A · Tests for the confusion matrix
- Theory 4 · Fixtures and parametrize
- Theory 5 · What to test in data science code
- Hands-on B · Tests for the retention cost, and a bug fixed test-first

---

## Setup

pytest runs from the terminal, not inside a notebook. To show its output here anyway, the two helpers below run
pytest in a separate process, exactly as the terminal would: `run_pytest_on` writes example test files into a
temporary folder first, `run_pytest_in_repo` runs it in the repository root, and `check_playground` runs the test
files you write in the exercises. You do not need to understand the helpers; they are plumbing for the notebook.

In [ ]:
import os
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

REPO = Path("..").resolve()
# No __pycache__ or .pytest_cache folders left behind by the demos
QUIET_ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}


def run_pytest_on(files: dict[str, str], *args: str) -> None:
    """Write {filename: code} to a temporary folder, run pytest there, print the report."""
    with tempfile.TemporaryDirectory() as tmp:
        for name, code in files.items():
            (Path(tmp) / name).write_text(textwrap.dedent(code))
        result = subprocess.run(
            [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "--color=no", "-q", *args],
            cwd=tmp,
            capture_output=True,
            text=True,
            env=QUIET_ENV,
        )
    print(result.stdout.replace(tmp, "<tmp>"))


def run_pytest_in_repo(*args: str) -> None:
    """Run pytest in the repository root, like `uv run pytest ...` in a terminal."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "--color=no", *args],
        cwd=REPO,
        capture_output=True,
        text=True,
        env=QUIET_ENV,
    )
    print(result.stdout[-4000:])

In [ ]:
def check_playground(test_file: str, min_tests: int, must_contain: str = "") -> None:
    path = REPO / "playground" / test_file
    assert path.exists(), f"Create {path.relative_to(REPO)} first"
    if must_contain:
        assert must_contain in path.read_text(), f"{test_file} should use {must_contain}"
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "--color=no"]
        + [f"playground/{test_file}"],
        cwd=REPO,
        capture_output=True,
        text=True,
        env=QUIET_ENV,
    )
    print(result.stdout[-2000:])
    assert result.returncode == 0, "Some tests fail (see above)"
    passed = int(result.stdout.split(" passed")[0].split()[-1])
    assert passed >= min_tests, f"Expected at least {min_tests} tests, found {passed}"
    print("All checks passed")

---

## Theory 1 · Tests live in files, pytest runs them

pytest is not something you run inside a notebook. In a project, tests are ordinary Python files:

- files named `test_*.py`, usually in a `tests/` folder next to `src/`;
- inside them, functions named `test_*`;
- each test uses a plain `assert` to state what must be true.

From the terminal, in the project root:

```bash
uv run pytest                       # run every test in tests/
uv run pytest tests/test_data.py    # one file
uv run pytest -k unseen             # only tests whose name contains "unseen"
uv run pytest -x                    # stop at the first failure
uv run pytest --lf                  # rerun only the tests that failed last time
```

pytest finds the files, calls every test function, and reports a dot for each pass and an `F` for each failure.

---

## Theory 2 · A first test: Arrange, Act, Assert

Take a small data-cleaning function. A telecom exports `TotalCharges` as text; a blank value is fine for a new
customer (tenure 0, not billed yet, so 0) and an error for anybody else. Here is the function and a test file for it.

Each test follows the same three steps, often called **Arrange, Act, Assert**: build the input, call the code,
check the result. The test name says what behaviour is expected, so a failure report reads like a sentence.

In [ ]:
code_under_test = """
import pandas as pd


def parse_total_charges(df: pd.DataFrame) -> pd.Series:
    # .str.strip() removes surrounding spaces; to_numeric(errors="coerce") turns text that is
    # not a number (the blanks) into NaN instead of failing
    total = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
    # .isna() is True where a value is missing; & combines two True/False columns row by row
    unexplained = total.isna() & (df["tenure"] > 0)
    if unexplained.any():  # .any(): is at least one row True?
        raise ValueError(f"TotalCharges missing for {int(unexplained.sum())} customers with tenure > 0")
    return total.fillna(0.0)  # replace the remaining NaN (tenure 0) with 0.0
"""

tests = """
import pandas as pd
import pytest

from charges import parse_total_charges


def test_text_values_become_numbers():
    df = pd.DataFrame({"tenure": [1, 34], "TotalCharges": ["29.85", "1889.5"]})   # Arrange
    result = parse_total_charges(df)                                               # Act
    assert result.tolist() == [29.85, 1889.5]                                      # Assert


def test_blank_for_new_customer_becomes_zero():
    df = pd.DataFrame({"tenure": [0], "TotalCharges": [" "]})
    assert parse_total_charges(df).tolist() == [0.0]


def test_blank_for_existing_customer_is_an_error():
    df = pd.DataFrame({"tenure": [12], "TotalCharges": [" "]})
    with pytest.raises(ValueError, match="tenure > 0"):
        parse_total_charges(df)
"""

run_pytest_on({"charges.py": code_under_test, "test_charges.py": tests})

Three dots, three passes. Now imagine a colleague "simplifies" the function and replaces every blank with 0. The
code still runs, the notebook still trains. Only the test notices:

In [ ]:
simplified = code_under_test.replace(
    'total = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")',
    'total = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce").fillna(0.0)',
)
run_pytest_on({"charges.py": simplified, "test_charges.py": tests})

Read the failure report from the bottom up: the short summary names the test, and the section above it shows
the line in the test that failed. pytest rewrites plain `assert` statements so that you see the values involved,
without any special assertion methods.

---

## Theory 3 · Comparing floats and expecting errors

The tests below import from `churn`, the reference package in `src/churn`. It is installed in this environment like pandas; notebook 11 shows how.

Two tools you will use constantly:

- `pytest.approx`: floating point arithmetic is not exact (`0.1 + 0.2 != 0.3`). Compare with a tolerance:
  `assert precision == pytest.approx(0.75)`.
- `pytest.raises`: a test that expects an exception. Use `match=` to check the message too, so the test does not
  pass because of some *other* error of the same type.

In [ ]:
print(0.1 + 0.2 == 0.3)

approx_and_raises = """
import pytest
from churn.evaluation import ConfusionMatrix


def test_precision():
    cm = ConfusionMatrix.from_predictions([0, 1, 1, 1], [1, 1, 1, 0])
    assert cm.precision == pytest.approx(2 / 3)


def test_shapes_must_match():
    with pytest.raises(ValueError, match="shape"):
        ConfusionMatrix.from_predictions([0, 1, 1], [0, 1])
"""
run_pytest_on({"test_approx.py": approx_and_raises})

Notice the import: `from churn.evaluation import ConfusionMatrix`. Because `uv sync` installed the package, tests
anywhere can import it. That is one of the reasons code that should be tested belongs in a package, not in a
notebook (notebook 11).

> **Check your understanding**
>
> This test passes. Does that tell you the scaler is correct?
>
> ```python
> def test_scaler_uses_training_mean():
>     scaler = Standardizer(["tenure"]).fit(X_train)
>     assert scaler._mean["tenure"] == scaler._mean["tenure"]
> ```
>
> <details><summary><b>Show answer</b></summary>
>
> We cannot tell: the assertion compares a value with itself, so it passes for any implementation. A test must
> compare the result with something computed **independently**, ideally a value you know by hand
> (`X_train["tenure"].mean()`, or a tiny example where the mean is obviously 10). It should also test behaviour
> through the public interface (`transform` gives mean 0 on training data), not private attributes such as
> `_mean`, which may be renamed tomorrow.
> </details>

In [ ]:
# Predict the output first, then run this cell
tautology = """
import pandas as pd
from churn.features import Standardizer


class BrokenStandardizer:
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X):
        self._mean = X[self.columns].mean() * 0  # deliberately wrong
        return self


def test_real_scaler_uses_training_mean():
    X_train = pd.DataFrame({"tenure": [1, 12, 50]})
    scaler = Standardizer(["tenure"]).fit(X_train)
    assert scaler._mean["tenure"] == scaler._mean["tenure"]


def test_broken_scaler_uses_training_mean():
    X_train = pd.DataFrame({"tenure": [1, 12, 50]})
    scaler = BrokenStandardizer(["tenure"]).fit(X_train)
    assert scaler._mean["tenure"] == scaler._mean["tenure"]
"""
run_pytest_on({"test_tautology.py": tautology}, "-v")

---

## Hands-on A · Tests for the confusion matrix

From here on you write real test files, as you would in a project, for classes of the reference package `churn`.
Create them in the `playground/` folder at the repository root (it is ignored by git, so you can experiment
freely) and run them in a terminal:

```bash
uv run pytest playground
```

The check cell after each exercise runs the same command for you.

### Exercise A.1

*About 15 minutes, including the discussion.*

Create `playground/test_confusion.py` with at least three tests for `churn.evaluation.ConfusionMatrix`:

1. `from_predictions` counts correctly on a small example you can verify by hand (write the expected `tn, fp, fn, tp`).
2. When the model flags nobody, `precision` is `0.0` instead of a `ZeroDivisionError`.
3. `y_true` and `y_pred` of different lengths raise a `ValueError` (use `pytest.raises`).

Use `pytest.approx` for float comparisons.

**What your code should do.**

- A file `playground/test_confusion.py` with at least three functions whose names start with `test_`; pytest
  must report all of them as passed.
- The behaviour you are testing: `ConfusionMatrix.from_predictions(y_true, y_pred)` takes two lists of 0/1 labels
  of the same length and returns an object with the integer attributes `tn`, `fp`, `fn`, `tp`.
- `precision` and `recall` are properties (no parentheses) that return floats; `precision` is `0.0` when the
  model flagged nobody.
- Lists of different lengths raise a `ValueError` whose message contains the word `shape`.
- No pandas needed: plain Python lists such as `[0, 1, 1]` are enough as input.

**Discuss.** Your test pins "nobody flagged" to a precision of `0.0`. Is that the right answer, or should it be `NaN` or an error? Which choice would you rather discover through a failing test than in a dashboard that quietly shows 0 %?

In [ ]:
# Check for exercise A.1: run after solving it
check_playground("test_confusion.py", min_tests=3)

<details><summary><b>Show hint</b></summary>

Start the file with `import pytest` and `from churn.evaluation import ConfusionMatrix`. For test 1, write the two
lists one above the other in a comment, so the reader can count along:

```python
#      y_true: 0  0  1  1  1
#      y_pred: 0  1  1  1  0
```
</details>

<details><summary><b>Show solution</b></summary>

```python
# playground/test_confusion.py
import pytest

from churn.evaluation import ConfusionMatrix


def test_counts_on_small_example():
    #      y_true: 0  0  1  1  1
    #      y_pred: 0  1  1  1  0
    cm = ConfusionMatrix.from_predictions([0, 0, 1, 1, 1], [0, 1, 1, 1, 0])
    assert (cm.tn, cm.fp, cm.fn, cm.tp) == (1, 1, 1, 2)
    assert cm.recall == pytest.approx(2 / 3)


def test_precision_is_zero_when_nobody_is_flagged():
    cm = ConfusionMatrix.from_predictions([0, 1, 1], [0, 0, 0])
    assert cm.precision == 0.0


def test_different_lengths_are_rejected():
    with pytest.raises(ValueError, match="shape"):
        ConfusionMatrix.from_predictions([0, 1, 1], [0, 1])
```
</details>

---

## Theory 4 · Fixtures and parametrize

### Fixtures: shared test data

Many tests need the same input, for example a small table of customers. Copying it into every test is tedious and
hides what each test is about. A **fixture** is a function decorated with `@pytest.fixture` that builds the input.
A test asks for it simply by naming it as a parameter; pytest calls the fixture and passes the result in.

Two rules:

- **Never call a fixture yourself.** pytest calls it (calling it directly raises an error). Each test gets a fresh
  result, so one test cannot spoil the data for the next.
- Fixtures used by several test files go into a file called `conftest.py`. pytest loads it automatically; no
  import needed.

This repository's own `tests/conftest.py` defines a fixture with six hand-written customers that cover the tricky
cases:

In [ ]:
conftest = (REPO / "tests" / "conftest.py").read_text()
print(conftest[conftest.index("@pytest.fixture") : conftest.index('"Churn"') + 60])

And a test in `tests/test_data.py` uses it by name:

```python
def test_blank_total_charges_of_new_customer_becomes_zero(raw_customers):
    df = clean_customers(raw_customers)
    assert df.loc["A-2", "TotalCharges"] == 0.0
```

Two built-in fixtures are worth knowing: `tmp_path` (a fresh temporary folder, for testing code that writes
files) and `caplog` (captures log messages, so you can assert that a warning was logged).

### parametrize: one test, many cases

When the same check applies to several inputs, `@pytest.mark.parametrize` runs the test once per case. Each case
is reported separately, so you see exactly which input failed.

In [ ]:
parametrized = """
import pytest
from churn.config import ExperimentConfig
from churn.errors import ConfigError


@pytest.mark.parametrize(
    "kwargs",
    [
        {"test_size": 0.0},
        {"test_size": 1.5},
        {"threshold": 1.0},
        {"model": "xgboost"},
    ],
)
def test_invalid_config_is_rejected(kwargs):
    with pytest.raises(ConfigError):
        ExperimentConfig(**kwargs)
"""
run_pytest_on({"test_config.py": parametrized}, "-v")

> **Check your understanding**
>
> A test file contains one plain test, and one test decorated with `@pytest.mark.parametrize` with three cases,
> plus a fixture. How many tests does pytest report?
>
> <details><summary><b>Show answer</b></summary>
>
> Four. Each parametrize case is a separate test; the plain test is one more. Fixtures are not tests, they only
> provide input.
> </details>

In [ ]:
# Predict the output first, then run this cell
counting = """
import pytest


@pytest.fixture
def churn_labels():
    return [0, 1, 1, 0]


def test_labels_are_binary(churn_labels):
    assert set(churn_labels) <= {0, 1}


@pytest.mark.parametrize("threshold", [0.3, 0.5, 0.7])
def test_threshold_is_a_probability(threshold):
    assert 0 < threshold < 1
"""
run_pytest_on({"test_counting.py": counting}, "-v")

---

## Theory 5 · What to test in data science code

You cannot test that a model is "good" with a unit test. You can test that the code around it does what you
think. Good candidates:

| What | Example from `tests/` |
|---|---|
| Data cleaning rules | blank `TotalCharges` becomes 0 only for tenure 0 (`test_data.py`) |
| Transformer invariants | fit learns from training data only; `transform` does not modify its input; output columns do not depend on the batch (`test_features.py`) |
| Errors for bad input | missing column raises `SchemaError` naming it |
| Shapes and ranges | one prediction per customer, probabilities in [0, 1] (`test_model.py`) |
| Reproducibility | two runs with the same seed give the same result (`test_experiment.py`) |
| Known bugs | the warm-up bug as a **regression test**: a customer's score must not depend on who else is in the batch |
| Smoke test | the whole pipeline runs end to end on real data and is clearly better than chance |

Here is the regression test for the warm-up bug, from `tests/test_model.py`:

In [ ]:
test_model = (REPO / "tests" / "test_model.py").read_text()
start = test_model.index("def test_score_of_a_customer_does_not_depend_on_the_batch")
print(test_model[start : test_model.index("\n\n\n", start)])

What **not** to test:

- **Other people's libraries.** You do not need to check that `LogisticRegression` fits correctly; scikit-learn has
  its own tests. Test *your* code around it.
- **Exact metric values** such as `accuracy == 0.8123`. They change with every library update or data refresh, and
  the test breaks without anything being wrong. Test a sensible floor instead (`roc_auc > 0.75`).
- **Private details** (`_mean`, `_categories`). Test behaviour through the public interface, so you can refactor
  the inside freely.

Keep unit tests fast: tiny hand-made DataFrames run in milliseconds. A few slower end-to-end tests on the real data
are fine on top.

### Running a whole test suite, with coverage

Let us run this repository's suite, as you would with `uv run pytest` in the terminal. The `--cov` option (from
the `pytest-cov` plugin) also measures which lines of `src/churn` the tests execute. Lines that never run are
lines no test protects.

In [ ]:
coverage_file = Path(tempfile.gettempdir()) / "churn_workshop.coverage"
QUIET_ENV["COVERAGE_FILE"] = str(coverage_file)  # keep the .coverage data file out of the repo
run_pytest_in_repo("tests", "--cov=churn", "--cov-report=term-missing")

The `Missing` column lists line numbers that no test runs. 100 % coverage is not the goal (it says nothing about
whether the assertions are meaningful), but uncovered lines in important logic are worth a look.

---

## Hands-on B · Tests for the retention cost, and a bug fixed test-first

Now use fixtures and parametrize on a class with arithmetic you can check by hand, the `RetentionCost` metric
from notebook 03 (finished version in `src/churn/evaluation.py`). The challenge turns the warm-up notebook's
encoding bug into a regression test.

### Exercise B.1

*About 15 minutes, including the discussion.*

Create `playground/test_cost.py` for `churn.evaluation.RetentionCost`:

1. A fixture `cost` that returns `RetentionCost(offer_cost=10, customer_value=100, acceptance_rate=0.5)`, numbers
   small enough to compute by hand.
2. A parametrized test with at least three cases `(y_true, y_pred, expected_cost)`. Work out each expected value
   on paper first: every flagged customer costs an offer; a flagged churner is still lost with probability
   `1 - acceptance_rate`; an unflagged churner is lost for sure.
3. A test that flagging nobody costs exactly `cost.do_nothing(y_true)`.

**What your code should do.**

- A file `playground/test_cost.py` with a fixture named `cost`, one test decorated with
  `@pytest.mark.parametrize` that has at least three cases, and one more test. pytest must report at least 4
  passing tests (each parametrize case counts as one).
- The behaviour you are testing: `cost(y_true, y_pred)` takes two lists of 0/1 labels and returns a float. Every
  flagged customer (`y_pred` is 1) adds `offer_cost`; every flagged churner adds
  `(1 - acceptance_rate) * customer_value`; every churner who was not flagged adds `customer_value`.
- With the fixture's numbers, `cost([1], [1])` is `10 + 0.5 * 100 = 60`.
- `cost.do_nothing(y_true)` returns `customer_value` times the number of churners in `y_true`.

**Discuss.** You worked out the expected costs by hand. Why is that better than computing them in the test with the same formula the code uses? When do hand-computed expectations stop being practical, and what would you test instead?

In [ ]:
# Check for exercise B.1: run after solving it
check_playground("test_cost.py", min_tests=4, must_contain="parametrize")

<details><summary><b>Show solution</b></summary>

```python
# playground/test_cost.py
import pytest

from churn.evaluation import RetentionCost


@pytest.fixture
def cost():
    return RetentionCost(offer_cost=10, customer_value=100, acceptance_rate=0.5)


@pytest.mark.parametrize(
    ("y_true", "y_pred", "expected"),
    [
        ([0, 0], [0, 0], 0),        # nobody churns, nobody flagged
        ([0], [1], 10),             # one wasted offer
        ([1], [0], 100),            # one churner missed
        ([1], [1], 10 + 50),        # offer, then half the value still lost
        ([1, 0, 1], [1, 1, 0], 170),
    ],
)
def test_cost(cost, y_true, y_pred, expected):
    assert cost(y_true, y_pred) == pytest.approx(expected)


def test_flagging_nobody_equals_doing_nothing(cost):
    y_true = [1, 0, 1, 1]
    assert cost(y_true, [0, 0, 0, 0]) == pytest.approx(cost.do_nothing(y_true))
```
</details>

### Exercise B.2: fix a bug test-first

> **Challenge**

This is the encoding step of the warm-up notebook, extracted into a function:

```python
# playground/batch_encoding.py
import pandas as pd


def encode_contracts(df: pd.DataFrame) -> pd.DataFrame:
    return pd.get_dummies(df["Contract"], prefix="Contract").astype(int)
```

1. Save it as `playground/batch_encoding.py`.
2. Write `playground/test_batch_encoding.py` with a test that **reproduces the bug**: encoding a batch that only
   contains two-year customers must give the same columns (and the same row values) as encoding them together with
   other customers. Run it and watch it fail. Now you have proof of the bug.
3. Fix `encode_contracts` so the test passes. The fix must not depend on the batch, so the possible categories
   have to come from somewhere else than the batch itself (an argument, a constant, or a learned object as in
   notebook 04).

Writing the failing test first guarantees that the bug can never come back unnoticed: it becomes a
**regression test**.

(Alternative challenge: run the coverage report from Theory 5, find a line in `src/churn` that no test executes,
and write a test in `playground/` that covers it.)

**What your code should do.**

- `encode_contracts(df)` receives a DataFrame with a text column `Contract` and returns a new DataFrame with one
  0/1 integer column per contract type, named like `Contract_Two year`, with the same index (row labels) as `df`.
- The columns (names and order) are the same whatever contract types happen to be in `df`: a batch with only
  two-year customers still gets all three columns, the missing ones filled with 0.
- Your test fails for the original function and passes for the fixed one.

**pandas and NumPy you need.**

- `pd.DataFrame({"Contract": ["Two year", "One year"]})`: a small table to use in a test.
- `df[df["Contract"] == "Two year"]`: keeps only the rows where the condition is True; the row labels are kept.
- `bigger.loc[small.index]`: selects rows of `bigger` by the row labels of `small`, to compare the same customers.
- `pd.get_dummies(df["Contract"], prefix="Contract").astype(int)`: one 0/1 column per value **present in this
  data** (`.astype(int)` turns True/False into 1/0).
- `pd.testing.assert_frame_equal(a, b)`: fails with a readable report if two DataFrames differ in columns, index
  or values.
- `pd.Categorical(values, categories=["Month-to-month", "One year", "Two year"])`: tells pandas the full list of
  possible values, so `get_dummies` creates a column for each of them even if it does not occur.

**Discuss.** Which of the three ways to provide the categories would you choose for production code? What happens with each one on the day marketing launches a "Three year" contract, and which failure would you notice first?

In [ ]:
# Check for exercise B.2: run after solving it
check_playground("test_batch_encoding.py", min_tests=1)

<details><summary><b>Show solution</b></summary>

```python
# playground/test_batch_encoding.py
import pandas as pd

from batch_encoding import encode_contracts


def test_encoding_does_not_depend_on_the_batch():
    customers = pd.DataFrame({"Contract": ["Two year", "Month-to-month", "One year", "Two year"]})
    two_year_only = customers[customers["Contract"] == "Two year"]

    together = encode_contracts(customers).loc[two_year_only.index]
    alone = encode_contracts(two_year_only)

    pd.testing.assert_frame_equal(alone, together)
```

With the original code this fails: the batch with only two-year customers produces a single column
`Contract_Two year`. A fix that takes the categories as a constant:

```python
# playground/batch_encoding.py
import pandas as pd

CONTRACT_TYPES = ["Month-to-month", "One year", "Two year"]


def encode_contracts(df: pd.DataFrame, categories: list[str] = CONTRACT_TYPES) -> pd.DataFrame:
    contracts = pd.Categorical(df["Contract"], categories=categories)
    return pd.get_dummies(contracts, prefix="Contract").astype(int).set_axis(df.index)
```

A constant is fine for a column whose values are fixed by the business (three contract types). For columns whose
values change over time, the categories should be **learned** from the training data and stored with the model, as
the `OneHotEncoder` in `src/churn/features.py` does. Then training and scoring can never disagree.
</details>

---

## Summary

- Tests are `test_*` functions in `test_*.py` files with plain `assert`s; run them with `uv run pytest`.
- Structure each test as Arrange, Act, Assert, and name it after the behaviour it checks.
- `pytest.approx` for floats, `pytest.raises(..., match=...)` for expected errors.
- Fixtures provide test data; pytest injects them by parameter name. Never call them yourself. Shared ones go in
  `conftest.py`.
- `@pytest.mark.parametrize` runs one test over many cases.
- In DS code, test cleaning rules, transformer invariants, error handling, shapes, reproducibility and every bug you
  fix. Do not test library internals or exact metric values.

**Common pitfalls:** tests that compare a value with itself; tests that depend on each other's leftovers; asserting
on private attributes; huge test data where four rows would show the point; never running the tests.

---

## Going further

- Mocking: `unittest.mock.patch` replaces a slow or external dependency (a database, an API) during a test. With
  dependency injection (notebook 08) you often do not need it: pass an `InMemoryLoader` instead.
- Property-based testing with Hypothesis: generate hundreds of random inputs
  and check invariants such as "precision is always between 0 and 1".
- Run notebooks as tests with nbmake.
- Run the tests automatically on every push with GitHub Actions.

---

## Sources and further reading

**Sources.** This notebook is based on:

- pytest documentation: [Get started](https://docs.pytest.org/en/stable/getting-started.html), [fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html), [parametrize](https://docs.pytest.org/en/stable/how-to/parametrize.html), [monkeypatch](https://docs.pytest.org/en/stable/how-to/monkeypatch.html).
- [Breck et al. (2017), *The ML Test Score: A Rubric for ML Production Readiness and Technical Debt Reduction*](https://research.google/pubs/the-ml-test-score-a-rubric-for-ml-production-readiness-and-technical-debt-reduction/), Google. A checklist of tests for data, model and infrastructure.

**For self-study.** Where to go next if you want to learn more:

- [Real Python: pytest Tutorial, Effective Python Testing](https://realpython.com/pytest-python-testing/) (article, beginner to intermediate).
- [Jeremy Jordan, *Effective testing for machine learning systems*](https://www.jeremyjordan.me/testing-ml/) and [Eugene Yan, *How to Test Machine Learning Code and Systems*](https://eugeneyan.com/writing/testing-ml/) (blog posts, intermediate): what to test in ML code, with examples.
- [Brian Okken, *Python Testing with pytest*, 2nd edition](https://pragprog.com/titles/bopytest2/python-testing-with-pytest-second-edition/) (book, intermediate): the standard book on pytest.
- [Hypothesis](https://hypothesis.readthedocs.io/) for property-based testing and [nbmake](https://github.com/treebeardtech/nbmake) for testing notebooks (library docs).
- [GitHub Actions: Building and testing Python](https://docs.github.com/en/actions/use-cases-and-examples/building-and-testing/building-and-testing-python) (guide): run your tests on every push.

---

[← 09 · Exceptions and logging](09_exceptions_and_logging.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [11 · From notebook to package →](11_from_notebook_to_package.ipynb)